# Generation Fuel Source

In [ ]:
from pathlib import Path
import traceback
import urllib.request
import nemosis
import pandas as pd


Fetch and aggregate regional generation by fuel type from DISPATCH_UNIT_SCADA.

In [ ]:
def main():
    RAW_DIR = Path("Pre_processing/temporary_cache")
    REGION_TO_SUFFIX = {"NSW1": "nsw", "QLD1": "qld", "VIC1": "vic", "SA1": "sa"}
    processed_path = Path("Processed_data/8_generation_fuel.csv")

    # --- Build fuel mapping ---
    cache_path = RAW_DIR / "NEM Registration and Exemption List.xlsx"
    url = "https://www.aemo.com.au/-/media/files/electricity/nem/participant_information/nem-registration-and-exemption-list.xlsx"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(cache_path, "wb") as f:
        f.write(resp.read())
    try:
        df = pd.read_excel(cache_path, sheet_name="PU and Scheduled Loads", engine="openpyxl", header=0)
    finally:
        cache_path.unlink(missing_ok=True)

    df = df[df["Region"].isin(REGION_TO_SUFFIX)].copy()
    if "Dispatch Type" in df.columns:
        df = df[df["Dispatch Type"].astype(str).str.upper().isin(
            ["GENERATOR", "BIDIRECTIONAL", "BIDIRECTIONAL_UNIT", "GENERATING UNIT", "BIDIRECTIONAL UNIT"]
        )]
    df = df[df["DUID"].notna()].copy()
    df["DUID"] = df["DUID"].astype(str).str.strip()
    df = df[df["DUID"] != ""].drop_duplicates(subset=["DUID"], keep="last")

    duid_to_col, battery_duids = {}, []
    for _, row in df.iterrows():
        cols = ["Fuel Source - Primary", "Fuel Source - Descriptor", "Technology Type - Primary", "Technology Type - Descriptor"]
        text = " | ".join("" if pd.isna(v) else str(v).strip().lower() for v in (row.get(c) for c in cols))
        if "battery" in text:       bucket = "battery"
        elif "wind" in text:        bucket = "wind"
        elif "solar" in text or "pv" in text: bucket = "solar"
        elif "hydro" in text:       bucket = "hydro"
        elif any(t in text for t in ["gas", "diesel", "distillate"]): bucket = "gas"
        else: continue
        suffix = REGION_TO_SUFFIX.get(row["Region"])
        if suffix is None:
            continue
        duid = str(row["DUID"])
        duid_to_col[duid] = f"battery_mw_{suffix}" if bucket == "battery" else f"{bucket}_mw_{suffix}"
        if bucket == "battery":
            battery_duids.append(duid)

    all_duids = sorted(duid_to_col.keys())
    print(f"  {len(all_duids)} DUIDs mapped.", flush=True)

    # --- Skip already processed months ---
    already_processed = set()
    if processed_path.exists():
        existing_idx = pd.read_csv(processed_path, usecols=[0], parse_dates=True).iloc[:, 0]
        already_processed = set(pd.to_datetime(existing_idx).dt.to_period("M"))
        print(f"  Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)

    # --- Process each month ---
    total_months = pd.date_range("2018/01/01", pd.Timestamp("2026/01/01") - pd.Timedelta(days=1), freq="MS")
    for i, month in enumerate(total_months):
        if month.to_period("M") in already_processed:
            print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping (already processed).", flush=True)
            continue
        start = month.strftime("%Y/%m/%d %H:%M:%S")
        end   = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M:%S")
        print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching...", flush=True)
        try:
            chunk = nemosis.dynamic_data_compiler(
                start_time=start, end_time=end,
                table_name="DISPATCH_UNIT_SCADA",
                raw_data_location=str(RAW_DIR),
                select_columns=["SETTLEMENTDATE", "DUID", "SCADAVALUE"],
                filter_cols=["DUID"], filter_values=[all_duids],
                fformat="feather", keep_csv=False,
            )
            chunk["SETTLEMENTDATE"] = pd.to_datetime(chunk["SETTLEMENTDATE"])
            chunk["SCADAVALUE"]     = pd.to_numeric(chunk["SCADAVALUE"], errors="coerce")
            chunk["_col"]           = chunk["DUID"].map(duid_to_col)
            chunk = chunk[chunk["_col"].notna()].copy()

            chunk["_target"] = chunk["_col"]
            chunk["_value"]  = chunk["SCADAVALUE"].clip(lower=0)
            is_battery = chunk["DUID"].isin(battery_duids)
            if is_battery.any():
                charge    = is_battery & (chunk["SCADAVALUE"] < 0)
                discharge = is_battery & (chunk["SCADAVALUE"] >= 0)
                chunk.loc[charge,    "_target"] = chunk.loc[charge,    "_col"].str.replace("battery_mw_", "battery_charge_mw_",    regex=False)
                chunk.loc[charge,    "_value"]  = -chunk.loc[charge,   "SCADAVALUE"]
                chunk.loc[discharge, "_target"] = chunk.loc[discharge, "_col"].str.replace("battery_mw_", "battery_discharge_mw_", regex=False)
                chunk.loc[discharge, "_value"]  = chunk.loc[discharge, "SCADAVALUE"].clip(lower=0)

            month_df = chunk.groupby(["SETTLEMENTDATE", "_target"])["_value"].sum().unstack("_target")
            month_df = month_df.resample("5min").mean().fillna(0)

            for f in RAW_DIR.glob(f"*{month.strftime('%Y%m')}*"):
                if f.is_file():
                    f.unlink()

            if processed_path.exists():
                existing = pd.read_csv(processed_path, index_col=0, parse_dates=True)
                existing.index = pd.to_datetime(existing.index, format="mixed")
                month_df = pd.concat([existing, month_df])
                month_df = month_df[~month_df.index.duplicated(keep="last")].sort_index()
            month_df.index.name = "SETTLEMENTDATE"
            month_df.to_csv(processed_path)
            already_processed.add(month.to_period("M"))
            print(f"  {month:%Y-%m} saved.", flush=True)
        except Exception:
            print(f"  WARNING: {month:%Y-%m} failed:\n{traceback.format_exc()}", flush=True)

    return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail() if processed_path.exists() else None

main()


  Found 97 already-processed month(s) — will skip.
Building fuel mapping...
  473 DUIDs mapped.
  1/96 2018-01 skipping (already processed).
  2/96 2018-02 skipping (already processed).
  3/96 2018-03 skipping (already processed).
  4/96 2018-04 skipping (already processed).
  5/96 2018-05 skipping (already processed).
  6/96 2018-06 skipping (already processed).
  7/96 2018-07 skipping (already processed).
  8/96 2018-08 skipping (already processed).
  9/96 2018-09 skipping (already processed).
 10/96 2018-10 skipping (already processed).
 11/96 2018-11 skipping (already processed).
 12/96 2018-12 skipping (already processed).
 13/96 2019-01 skipping (already processed).
 14/96 2019-02 skipping (already processed).
 15/96 2019-03 skipping (already processed).
 16/96 2019-04 skipping (already processed).
 17/96 2019-05 skipping (already processed).
 18/96 2019-06 skipping (already processed).
 19/96 2019-07 skipping (already processed).
 20/96 2019-08 skipping (already processed).
 21/

,gas_mw_nsw,gas_mw_qld,gas_mw_sa,gas_mw_vic,hydro_mw_nsw,hydro_mw_qld,hydro_mw_vic,solar_mw_nsw,solar_mw_qld,wind_mw_nsw,...,solar_mw_sa,wind_mw_qld,battery_discharge_mw_qld,battery_charge_mw_nsw,battery_charge_mw_qld,battery_charge_mw_sa,battery_discharge_mw_nsw,battery_discharge_mw_sa,battery_discharge_mw_vic,battery_charge_mw_vic
SETTLEMENTDATE,,,,,,,,,,,,,,,,,,,,,
2025-12-31 23:40:00,86.771919,156.365349,43.10546,0.0,113.28961,318.32999,95.98,0.017216,1.16495,1640.267476,...,0.0,842.35040,133.86300,0.80920,1.56962,0.00000,127.54788,456.45114,374.17578,1.45959
2025-12-31 23:45:00,87.571920,156.233351,41.10546,0.0,83.50161,319.27999,92.66,0.016216,1.15862,1648.594747,...,0.0,789.91940,119.62123,1.14157,8.41681,0.23387,233.07663,413.67138,350.05945,1.60173
2025-12-31 23:50:00,86.561918,155.575341,42.10546,0.0,82.05761,319.67000,92.62,0.016216,1.15000,1636.571901,...,0.0,729.00139,120.65746,2.24656,2.85329,0.17618,258.30877,399.69722,338.70366,1.27000
2025-12-31 23:55:00,86.801917,155.135339,41.10546,0.0,81.87451,318.45250,92.38,0.016216,1.15000,1671.306052,...,0.0,698.62979,113.48361,6.78110,31.12263,0.81516,250.18228,278.79363,349.52897,1.94413
2026-01-01 00:00:00,86.921919,156.630350,41.10546,0.0,82.19613,318.06249,92.26,0.017216,1.15241,1673.534913,...,0.0,684.93010,90.89073,0.60826,7.70418,0.00966,261.88294,364.78201,356.87402,1.18000
